<a href="https://colab.research.google.com/github/vladislavpykhteev/HSE_Vladislav_Pykhteev/blob/main/fianl/final_hw_data/final_hw_efrsb_%D0%9F%D1%8B%D1%85%D1%82%D0%B5%D0%B5%D0%B2_%D0%92_%D0%A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ Итоговое домашнее задание
# «Аналитик реестра — Обработка данных ЕФРСБ»





Вы — юрист, анализирующий данные из **Единого федерального реестра сведений о банкротстве (ЕФРСБ)**.
Вам поступила выгрузка в формате JSON с информацией о сообщениях по делам о банкротстве за 2025 год.

**Ваша задача** — извлечь ключевые данные, очистить от мусора, сопоставить с реестром организаций
и сформировать аналитический отчёт для руководства.

---
### 📋 Как выполнять и сдавать задание

**Где выполнять:**
- **Вариант 1 (рекомендуется):** Загрузите файл `.ipynb` в [Google Colab](https://colab.research.google.com/), загрузите папку `final_hw_data/` в среду выполнения (например, через боковую панель «Файлы» или смонтировав Google Drive).
- **Вариант 2:** Работайте локально в Jupyter Notebook / JupyterLab. Убедитесь, что папка `final_hw_data/` находится в той же директории, что и файл задания.

**Как сдавать на проверку:**
- Отправьте **ссылку на Google Colab** с общим доступом («Любой, у кого есть ссылка — комментатор»), **ИЛИ**
- Отправьте **ссылку на GitHub-репозиторий**, в котором лежит выполненный `.ipynb`-файл и папка `final_hw_data/` с результатами.

> ⚠️ Убедитесь, что ссылка открывается в режиме просмотра и все ячейки выполнены (результаты видны).

---

### 📦 Исходные данные

В папке `final_hw_data/` находятся 3 файла:

| Файл | Описание |
|------|----------|
| `bankruptcy_messages.json` | Сообщения ЕФРСБ |
| `organizations.json` | Реестр организаций |
| `priority_cases.txt` | Номера приоритетных дел |



---
# 🟢 Загрузка и кросс-референс (0-2 балла)
---

### Задание 1.1 — Загрузка данных из файлов

Загрузите данные из трёх файлов:
- `bankruptcy_messages.json` → список `messages`
- `organizations.json` → список `organizations`
- `priority_cases.txt` → список `priority_case_numbers`

Выведите количество загруженных записей из каждого источника.

In [1]:
# Ваш код
"""
Подготовка - подгружаем исходные данные из GitHub-репозитория
"""
import os
import urllib.request
import json
# адрес файлов в репозитории
REPO_RAW = "https://raw.githubusercontent.com/vladislavpykhteev/HSE_Vladislav_Pykhteev/main/final"

# список файлов, которые нужны для всех заданий
SOURCE_FILES = [
    "bankruptcy_messages.json",
    "organizations.json",
    "priority_cases.txt",
]

# создаём папку (если её ещё нет)
os.makedirs("final_hw_data", exist_ok=True)

# проходим по списку и качаем те файлы, которых нет локально
for fileName in SOURCE_FILES:
    localPath = "final_hw_data/" + fileName
    if os.path.exists(localPath):
        print(f"  уже есть: {localPath}")
        continue
    url = REPO_RAW + "/" + fileName
    try:
        urllib.request.urlretrieve(url, localPath)
    except Exception as e:
        print(f"  ОШИБКА при скачивании {fileName}: {e}")

# путь к папке с данными
DATA_DIR = "final_hw_data"

# загружаем сообщения ЕФРСБ
with open(f"{DATA_DIR}/bankruptcy_messages.json", "r", encoding="utf-8") as f:
    messages = json.load(f)

# загружаем реестр организаций
with open(f"{DATA_DIR}/organizations.json", "r", encoding="utf-8") as f:
    organizations = json.load(f)

# загружаем приоритетные дела - txt-файл
with open(f"{DATA_DIR}/priority_cases.txt", "r", encoding="utf-8") as f:
    rawLines = f.read().splitlines()
    # очищаем строки от пробелов и убираем пустые
    priority_case_numbers = [line.strip() for line in rawLines if line.strip()]

print("Загружено сообщений ЕФРСБ:", len(messages))
print("Загружено организаций:    ", len(organizations))
print("Загружено приоритетных дел:", len(priority_case_numbers))

Загружено сообщений ЕФРСБ: 54
Загружено организаций:     30
Загружено приоритетных дел: 10


### Задание 1.2 — Словарь организаций

Создайте словарь `org_by_inn`, где ключ — ИНН (строка), а значение — словарь с данными об организации.
Используйте **dict comprehension**.

Выведите количество организаций в словаре и информацию по ИНН `"7701234567"`.

In [2]:
# Ваш код
# Ключ - ИНН (строка), значение - вся карточка организации
org_by_inn = {org["inn"]: org for org in organizations}

print("Количество организаций в словаре:", len(org_by_inn))
print("Сведения об организации с ИНН 7701234567:")
print(org_by_inn.get("7701234567"))

Количество организаций в словаре: 30
Сведения об организации с ИНН 7701234567:
{'inn': '7701234567', 'ogrn': '1027700123456', 'name': 'ООО "Альфа-Строй"', 'address': 'г. Москва, ул. Строителей, д. 15', 'region': 'Москва'}


### Задание 1.3 — Кросс-референс: связываем сообщения с организациями

Для каждого сообщения добавьте поле `org_name` (название организации) и `region`,
используя словарь `org_by_inn` и метод `.get()`.
Если организация не найдена — ставьте значение `None`.

Посчитайте, сколько сообщений удалось связать с организацией,
а сколько — не удалось.

In [3]:
# Ваш код
# Через .get() безопасно получаем словарь по ИНН
# Считаем сколько удалось и не удалось связать

linkedCount = 0
notLinkedCount = 0

for msg in messages:
    inn = msg.get("publisher_inn")
    org = org_by_inn.get(inn)

    if org is not None:
        msg["org_name"] = org.get("name")
        msg["region"] = org.get("region")
        linkedCount = linkedCount + 1
    else:
        msg["org_name"] = None
        msg["region"] = None
        notLinkedCount = notLinkedCount + 1

print("Связано с организацией:  ", linkedCount)
print("Не удалось связать:      ", notLinkedCount)

Связано с организацией:   51
Не удалось связать:       3


### Задание 1.4 — Множества и приоритетные дела

1. Создайте множество `priority_set` из списка `priority_case_numbers`.
2. Создайте множество `message_case_set` из номеров дел в `messages`
   (используйте list comprehension + filter для непустых номеров).
3. Найдите пересечение — номера дел, которые есть и в приоритетных, и в сообщениях (`&`).
4. Выведите результат.

In [4]:
# Ваш код
# Подзадача 1:
priority_set = set(priority_case_numbers)
# Подзадача 2:
message_case_set = set(
    [msg["case_number"] for msg in messages if msg.get("case_number")]
)
# Подзадача 3:
intersection = priority_set & message_case_set

print("Количество дел, которые есть и в приоритетных, и в сообщениях:", len(intersection))
print()
print("Список общих номеров:")
i = 1
for caseNumber in sorted(intersection):
    print(i,"-  ", caseNumber)
    i += 1

Количество дел, которые есть и в приоритетных, и в сообщениях: 10

Список общих номеров:
1 -   А60-11111/2025
2 -   А60-12345/2025
3 -   А60-22222/2025
4 -   А60-33333/2025
5 -   А60-44444/2025
6 -   А60-56789/2025
7 -   А60-66666/2025
8 -   А60-77777/2025
9 -   А60-88888/2025
10 -   А60-99999/2025


---
# 🟡 Очистка и валидация (0-3 балла)
---

### Задание 2.1 — Функция парсинга дат

Напишите функцию `parse_date(date_str)`, которая принимает строку с датой
и возвращает объект `datetime` или `None`, если распарсить не удалось.

Форматы для обработки:
- `"DD.MM.YYYY"` — `strptime`
- `"YYYY-MM-DD"` — `fromisoformat`
- `"DD месяца YYYY г."` — замена русских месяцев + `strptime`
- `"DD/MM/YYYY HH:MM"` — `strptime`

Обрабатывайте `None` и пустые строки через `try/except`.

In [18]:
# Ваш код
# Если на вход пришёл None, пустая строка или строка вида "ДАТА УТЕРЯНА", то возвращаем None

from datetime import datetime

# словарь для замены русских месяцев на номера нужен для дат по типу "19 апреля 2026 г."
RUSSIAN_MONTHS = {
    "января": "01", "февраля": "02", "марта": "03", "апреля": "04",
    "мая": "05", "июня": "06", "июля": "07", "августа": "08",
    "сентября": "09", "октября": "10", "ноября": "11", "декабря": "12",
}

# список шаблонов strptime, которые перебираем по очереди, сначала более частые форматы
DATE_FORMATS = [
    "%d.%m.%Y",         # Формат: 15.01.2025
    "%d/%m/%Y %H:%M",   # Формат: 03/04/2025 14:30
    "%Y-%m-%d",         # Формат: 2025-02-10
    "%d %m %Y",         # Формат: после замены русского месяца на номер
]

def parse_date(date_str):

    # сначала проверяем на None и пустую строку
    if date_str is None or not isinstance(date_str, str):
        return None

    s = date_str.strip()
    if s == "":
        return None

    # пробуем ISO без шаблона
    try:
        return datetime.fromisoformat(s)
    except ValueError:
        pass

    # проверяем текстовое название месяца, при успехе поиска - заменяем на номер
    sNormalized = s.lower().replace(" г.", "").replace(" г", "")
    for ruMonth, numMonth in RUSSIAN_MONTHS.items():
        if ruMonth in sNormalized:
            sNormalized = sNormalized.replace(ruMonth, numMonth)
            break

    # перебираем шаблоны по очереди
    for fmt in DATE_FORMATS:
        try:
            # пробуем сначала исходную строку, потом нормализованную
            return datetime.strptime(s, fmt)
        except ValueError:
            try:
                return datetime.strptime(sNormalized, fmt)
            except ValueError:
                continue

    # если ни один формат не подошёл, то возвращаем None
    return None

# проверка функции
print("Проверка работы функции parse_date() для различных типов:")
print("="*60)
print("Входные данные       ->  Результат")
print("="*60)
print("'20.04.2026'         ->", parse_date("20.04.2026"))
print("'2026-04-19'         ->", parse_date("2026-04-19"))
print("'18/04/2026 14:30'   ->", parse_date("18/04/2026 14:30"))
print("'17 апреля 2026 г.'  ->", parse_date("17 апреля 2026 г."))
print("''(пустая строка)    ->", parse_date(""))
print("None                 ->", parse_date(None))
print("'ДАТА УТЕРЯНА'       ->", parse_date("ДАТА УТЕРЯНА"))

Проверка работы функции parse_date() для различных типов:
Входные данные       ->  Результат
'20.04.2026'         -> 2026-04-20 00:00:00
'2026-04-19'         -> 2026-04-19 00:00:00
'18/04/2026 14:30'   -> 2026-04-18 14:30:00
'17 апреля 2026 г.'  -> 2026-04-17 00:00:00
''(пустая строка)    -> None
None                 -> None
'ДАТА УТЕРЯНА'       -> None


### Задание 2.2 — Функция валидации сообщения

Напишите функцию `validate_message(msg)`, которая:

1. Проверяет наличие **обязательных полей**: `publisher_inn`, `msg_text`, `date_published`, `type`, `case_number`.
   Если поле отсутствует — ошибка типа `KeyError`.
2. Проверяет, что **ИНН** состоит из 10 или 12 цифр (метод `.isdigit()`).
3. Парсит дату через `parse_date()`. Если дата не парсится — ошибка.
4. Проверяет, что **тип сообщения** — непустая строка.

Функция возвращает кортеж `(valid_msg, errors)`:
- `valid_msg` — словарь с очищенными данными (или `None`)
- `errors` — список строк с описанием ошибок

In [19]:
# Ваш код
REQUIRED_FIELDS = ["publisher_inn", "msg_text", "date_published", "type", "case_number"]

def validate_message(msg):
    errors = []
    cleaned = {}

    # проверка обязательных полей
    for field in REQUIRED_FIELDS:
        if field not in msg:
            # фиксируем ошибку
            # продолжаем обработку, чтобы собрать все ошибки сразу
            errors.append(f"Отсутсвие поля: отсутствует поле '{field}'")

    if errors:
        return None, errors

    # 2) проверка ИНН - 10 или 12 цифр
    inn = msg.get("publisher_inn")
    if inn is None or not str(inn).isdigit():
        errors.append("Некорректный ИНН: ИНН должен состоять только из цифр")
    elif len(str(inn)) not in (10, 12):
        errors.append(f"ИНН неверной длины: ИНН должен быть длиной 10 или 12 цифр, получено {len(str(inn))}")

    # 3) проверка даты через функцию из шага 2.1
    parsedDate = parse_date(msg.get("date_published"))
    if parsedDate is None:
        errors.append(f"Распознание даты: не удалось распознать дату '{msg.get('date_published')}'")

    # 4) првоерка типа сообщения на none и пустую строку
    msgType = msg.get("type")
    if msgType is None or not isinstance(msgType, str) or msgType.strip() == "":
        errors.append("Пустое поле: тип сообщения должен быть непустой строкой")

    # дополнительная проверка заполнения номера дела
    caseNumber = msg.get("case_number")
    if caseNumber is None or str(caseNumber).strip() == "":
        errors.append("Нет номера дела: номер дела должен быть непустой строкой")

    # возвращаем none, если были ошибки
    if errors:
        return None, errors

    # собираем сообщение
    cleaned = {
        "publisher_inn":  str(inn),
        "msg_text":       msg.get("msg_text", ""),
        "date_published": parsedDate,
        "type":           msgType.strip(),
        "case_number":    str(caseNumber).strip(),
        "org_name":       msg.get("org_name"),
        "region":         msg.get("region"),
    }
    return cleaned, []

# проверка функции
print()
print("Проверяем validate_message() на первом сообщении:")
sampleValid, sampleErr = validate_message(messages[0])
print("   Ошибки:", sampleErr)
print("   Сообщение:", sampleValid)


Проверяем validate_message() на первом сообщении:
   Ошибки: []
   Сообщение: {'publisher_inn': '7701234567', 'msg_text': 'Сообщение о введении процедуры наблюдения в отношении ООО "Альфа-Строй". Арбитражный управляющий Иванов Иван Иванович. Долг составляет 15 000 000 руб. АС Свердловской области.', 'date_published': datetime.datetime(2025, 1, 15, 0, 0), 'type': 'Введение процедуры', 'case_number': 'А60-12345/2025', 'org_name': 'ООО "Альфа-Строй"', 'region': 'Москва'}


### Задание 2.3 — Массовая валидация

Примените `validate_message()` ко всем сообщениям.
Разделите результат на два списка: `valid_messages` и `error_records`.

Соберите **статистику ошибок** по типам (сколько `KeyError`, `ValueError` и т.д.).

In [7]:
# Ваш код
valid_messages = []
error_records = []
errorStats = {}

for msg in messages:
    cleaned, errs = validate_message(msg)
    if cleaned is not None:
        valid_messages.append(cleaned)
    else:
        # сохраняем исходное сообщение и список ошибок
        error_records.append({
            "original": msg,
            "errors": errs,
        })
        # подсчитываем типы ошибок
        for e in errs:
            errType = e.split(":")[0]
            errorStats[errType] = errorStats.get(errType, 0) + 1

print("Всего сообщений:                   ", len(messages))
print("Колличество сообщений без ошибок:  ", len(valid_messages))
print("Колличество сообщений с ошибками:  ", len(error_records))
print("Статистика по типам ошибок:")
for errType, count in errorStats.items():
    print(f"- {errType}: {count}")

Всего сообщений:                    54
Колличество сообщений без ошибок:   48
Колличество сообщений с ошибками:   6
Статистика по типам ошибок:
- Некорректный ИНН: 1
- Распознание даты: 2
- Пустое поле: 1
- ИНН неверной длины: 1
- Нет номера дела: 1


---
# 🔵 Извлечение данных и аналитика (0-2 балла)
---

### Задание 3.1 — Извлечение сумм из текста

Напишите функцию `extract_amounts(text)`, которая ищет упоминания денежных сумм в тексте сообщения.

Используйте строковые методы.

Ищите по ключевым словам: `"руб."`, `"тыс. руб."`, `"млн руб."`.

Функция возвращает список строк — найденных сумм.

In [8]:
# Ваш код
# ключевые слова, запись по длине
KEYWORDS = ["млн руб.", "тыс. руб.", "руб."]

def extract_amounts(text):
    if not text or not isinstance(text, str):
        return []

    results = []
    # приводим к одному регистру
    workText = text

    for keyword in KEYWORDS:
        # ищем все вхождения ключевого слова
        startPos = 0
        while True:
            foundPos = workText.find(keyword, startPos)
            if foundPos == -1:
                break

            # берём 30 символов слева, чтобы не увеличивать диапозон поиска
            leftStart = max(0, foundPos - 30)
            leftPart = workText[leftStart:foundPos]

            # из левой части берём последние слова и двигаемся справа налево пока встречаем цифры, пробелы или точки
            collected = ""
            for ch in reversed(leftPart):
                if ch.isdigit() or ch in " .,":
                    collected = ch + collected
                else:
                    break

            amountStr = collected.strip() + " " + keyword
            amountStr = amountStr.strip()
            if amountStr and amountStr != keyword:
                results.append(amountStr)

            # сдвигаем стартовую позицию, чтобы найти следующее вхождение
            startPos = foundPos + len(keyword)

            # подменяем найденный фрагмент пробелами, чтобы не находить его повторно
            workText = workText[:foundPos] + (" " * len(keyword)) + workText[foundPos + len(keyword):]

    return results

# проверка функции
print("Проверка работы функции extract_amounts():")
testText1 = "Кредитор включен в реестр требований на сумму 2 млн руб., из которых: 1 815 000 руб. - основной долг 85 тыс. руб. - неустойка"
print("   Текст:   ", testText1)
print("   Найдены суммы: ", extract_amounts(testText1))

Проверка работы функции extract_amounts():
   Текст:    Кредитор включен в реестр требований на сумму 2 млн руб., из которых: 1 815 000 руб. - основной долг 85 тыс. руб. - неустойка
   Найдены суммы:  ['2 млн руб.', '85 тыс. руб.', '1 815 000 руб.']


### Задание 3.2 — Поиск упоминаний арбитражных судов

Напишите функцию `find_court_mentions(text)`, которая проверяет,
содержит ли текст упоминание арбитражного суда.

Ищите подстроку `"АС "` (с пробелом) в тексте через оператор `in`.
Верните `True` / `False`.

In [9]:
# Ваш код
def find_court_mentions(text):
    if not text or not isinstance(text, str):
        return False
    return "АС " in text

# проверка функции
print("Проверка работы функции find_court_mentions():")
print("="*60)
print("Входные данные                 ->  Результат")
print("="*60)
print("'АС Свердловской области'      ->", find_court_mentions("АС Свердловской области"))
print("'текст без суда'               ->", find_court_mentions("обычный текст без суда"))
print("'как же ассиметрично это'      ->", find_court_mentions("как же ассиметрично это"))
print("'В суде АС города Москвы'      ->", find_court_mentions("В суде АС города Москвы"))

Проверка работы функции find_court_mentions():
Входные данные                 ->  Результат
'АС Свердловской области'      -> True
'текст без суда'               -> False
'как же ассиметрично это'      -> False
'В суде АС города Москвы'      -> True


### Задание 3.3 — Извлечение ФИО арбитражных управляющих

Напишите функцию `extract_manager_name(text)`, которая ищет ФИО арбитражного управляющего.

Алгоритм:
1. Проверьте, содержит ли текст подстроку `"арбитражный управляющий"`.
2. Если да — найдите позицию этой подстроки, возьмите текст после неё.
3. С помощью `.split()` извлеките следующие 3 слова (ФИО).
4. Соедините через пробел и верните.
5. Если не найдено — верните `None`.

In [10]:
# Ваш код
KEYWORD_AM = "арбитражный управляющий"

def extract_manager_name(text):
    if not text or not isinstance(text, str):
        return None

    lowerText = text.lower()
    pos = lowerText.find(KEYWORD_AM)
    if pos == -1:
        return None

    # получаем текст после ключевого слова, но из исходной строки, чтобы регистр ФИО сохранился
    afterText = text[pos + len(KEYWORD_AM):]

    # разбиваем по пробелам и берём первые три слова
    words = afterText.split()
    if len(words) < 3:
        return None

    fio = words[0] + " " + words[1] + " " + words[2]

    # уберём знаки препинания на конце слов (например "Иванович.")
    fioClean = fio.strip(" .,;:")
    return fioClean


# проверка функции
print("Проверка работы функции extract_manager_name():")
testText3 = "Назначен арбитражный управляющий Пыхтеев Владислав Сергеевич, являющийся членом СРО"
print("   Текст:", testText3)
print("   ФИО арбитражного управляющего:  ", extract_manager_name(testText3))

Проверка работы функции extract_manager_name():
   Текст: Назначен арбитражный управляющий Пыхтеев Владислав Сергеевич, являющийся членом СРО
   ФИО арбитражного управляющего:   Пыхтеев Владислав Сергеевич


### Задание 3.4 — Обогащение данных

Для каждого **валидного** сообщения добавьте поля:
- `amounts` — список извлечённых сумм (функция `extract_amounts`)
- `has_court_mention` — `True`/`False` (функция `find_court_mentions`)
- `manager_name` — ФИО или `None` (функция `extract_manager_name`)

In [11]:
# Ваш код
for msg in valid_messages:
    text = msg.get("msg_text", "")
    msg["amounts"] = extract_amounts(text)
    msg["has_court_mention"] = find_court_mentions(text)
    msg["manager_name"] = extract_manager_name(text)

# проверка функции на первых трёх сообщениях для проверки
print("Проверка работы добавления полей для каждого валидного сообщения:")
print()
for m in valid_messages[:3]:
    print("Дело № ", m["case_number"])
    print("-  распознанные суммы:", m["amounts"])
    print("-  арбитражный суд:  ", m["has_court_mention"])
    print("-  ФИО арбитражного управляющего:  ", m["manager_name"])
    print()

Проверка работы добавления полей для каждого валидного сообщения:

Дело №  А60-12345/2025
-  распознанные суммы: ['15 000 000 руб.']
-  арбитражный суд:   True
-  ФИО арбитражного управляющего:   Иванов Иван Иванович

Дело №  А60-56789/2025
-  распознанные суммы: ['8 500 тыс. руб.']
-  арбитражный суд:   True
-  ФИО арбитражного управляющего:   None

Дело №  А60-11111/2025
-  распознанные суммы: ['42 млн руб.']
-  арбитражный суд:   False
-  ФИО арбитражного управляющего:   Петров Пётр Петрович



### Задание 3.5 — Аналитика

Постройте следующие показатели по **валидным** сообщениям:

1. **Количество сообщений по типам** — словарь `{тип: количество}`
2. **Количество сообщений по регионам** — словарь `{регион: количество}`, пропуская `None`
3. **Топ-5 публикаторов** по количеству сообщений — `sorted(key=lambda...)`
4. **Таймлайн** — сообщения, отсортированные по дате публикации

In [12]:
# Ваш код

# 1) Количество сообщений по типам
type_stats = {}
for m in valid_messages:
    t = m["type"]
    type_stats[t] = type_stats.get(t, 0) + 1

# 2) Количество сообщений по регионам
region_stats = {}
for m in valid_messages:
    region = m.get("region")
    if region is None:
        continue
    region_stats[region] = region_stats.get(region, 0) + 1

# 3) Топ-5 публикаторов по количеству сообщений
publisherStats = {}
for m in valid_messages:
    inn = m["publisher_inn"]
    publisherStats[inn] = publisherStats.get(inn, 0) + 1

top_publishers = sorted(
    publisherStats.items(),
    key=lambda pair: pair[1],
    reverse=True,
)[:5]

# 4) Таймлайн — сообщения, отсортированные по дате публикации
timeline = sorted(valid_messages, key=lambda m: m["date_published"])

# Выводим результаты
print()
print("Статистика по типам сообщений:")
for t, c in type_stats.items():
    print(f"- {t}: {c}")

print()
print("Статистика по регионам:")
for r, c in region_stats.items():
    print(f"- {r}: {c}")

print()
print("Топ-5 публикаторов:")
for inn, count in top_publishers:
    orgName = org_by_inn.get(inn, {}).get("name", "—")
    print(f"- ИНН {inn} ({orgName}): {count} сообщений")

print()
print("Таймлайн (первые 5 сообщений по дате):")
for m in timeline[:5]:
    dateStr = m["date_published"].strftime("%d.%m.%Y")
    print(f"- {dateStr} | {m['type']} | {m['case_number']}")


Статистика по типам сообщений:
- Введение процедуры: 8
- Собрание кредиторов: 3
- Завершение процедуры: 6
- Признание банкротом: 3
- Продажа имущества: 6
- Требование кредитора: 4
- Оспаривание сделки: 3
- Подача заявления: 3
- Отстранение управляющего: 1
- Мировое соглашение: 3
- Субсидиарная ответственность: 2
- Передача дела: 3
- Назначение управляющего: 2
- Жалоба на управляющего: 1

Статистика по регионам:
- Москва: 25
- Свердловская область: 6
- Санкт-Петербург: 5
- Краснодарский край: 3
- Челябинская область: 3
- Ростовская область: 2
- Владимирская область: 1
- Московская область: 2

Топ-5 публикаторов:
- ИНН 7701234567 (ООО "Альфа-Строй"): 3 сообщений
- ИНН 7706567890 (ООО "Тета-Консалтинг"): 3 сообщений
- ИНН 7702345678 (ЗАО "Бета-Инвест"): 2 сообщений
- ИНН 6658123450 (ООО "Гамма-Трейд"): 2 сообщений
- ИНН 7810345612 (ОАО "Дельта-Логистик"): 2 сообщений

Таймлайн (первые 5 сообщений по дате):
- 10.01.2025 | Завершение процедуры | А60-25814/2025
- 15.01.2025 | Введение проце

---
# 🟣 Отчётность (0-1 балл)
---

### Задание 4.1 — Подготовка данных для сохранения

Подготовьте данные для записи в JSON. Даты нужно преобразовать обратно в строки (`strftime`),
чтобы JSON был читаемым.

In [13]:
# Ваш код
valid_messages_for_json = []
for m in valid_messages:
    # делаем копию словаря
    copyMsg = dict(m)
    if isinstance(copyMsg.get("date_published"), datetime):
        copyMsg["date_published"] = copyMsg["date_published"].strftime("%d.%m.%Y")
    valid_messages_for_json.append(copyMsg)

# проверка обработки на первом сообщении
print("Количество сообщений, подготовленных для записи в JSON:  ", len(valid_messages_for_json))
print("Пример структуры одного сообщения: ", valid_messages_for_json[0])

Количество сообщений, подготовленных для записи в JSON:   48
Пример структуры одного сообщения:  {'publisher_inn': '7701234567', 'msg_text': 'Сообщение о введении процедуры наблюдения в отношении ООО "Альфа-Строй". Арбитражный управляющий Иванов Иван Иванович. Долг составляет 15 000 000 руб. АС Свердловской области.', 'date_published': '15.01.2025', 'type': 'Введение процедуры', 'case_number': 'А60-12345/2025', 'org_name': 'ООО "Альфа-Строй"', 'region': 'Москва', 'amounts': ['15 000 000 руб.'], 'has_court_mention': True, 'manager_name': 'Иванов Иван Иванович'}


### Задание 4.2 — Сохранение `analysis_results.json`

Сохраните файл `analysis_results.json` со следующей структурой:
```json
{
  "valid_messages": [...],
  "type_stats": {...},
  "region_stats": {...},
  "priority_messages": [...]
}
```

In [14]:
# Ваш код*
priority_messages = [m for m in valid_messages_for_json if m["case_number"] in priority_set]

analysisResult = {
    "valid_messages":    valid_messages_for_json,
    "type_stats":        type_stats,
    "region_stats":      region_stats,
    "priority_messages": priority_messages,
}

with open(f"{DATA_DIR}/analysis_results.json", "w", encoding="utf-8") as f:
    json.dump(analysisResult, f, ensure_ascii=False, indent=2)

print(f"Файл analysis_results.json сохранён локально /{DATA_DIR}")
print("   Количество сообщений без ошибок:   ", len(valid_messages_for_json))
print("   Количество приоритетных сообщений: ", len(priority_messages))

Файл analysis_results.json сохранён локально /final_hw_data
   Количество сообщений без ошибок:    48
   Количество приоритетных сообщений:  32


### Задание 4.3 — Сохранение `validation_errors.json`

Сохраните проблемные записи с описанием ошибок.

In [15]:
# Ваш код
# проблемные записи и описания ошибок сохранены в error_records
with open(f"{DATA_DIR}/validation_errors.json", "w", encoding="utf-8") as f:
    json.dump(error_records, f, ensure_ascii=False, indent=2)

print(f"Файл validation_errors.json сохранён локально /{DATA_DIR}")
print("   Количество сообщений с ошибоками:   ", len(error_records))

Файл validation_errors.json сохранён локально /final_hw_data
   Количество сообщений с ошибоками:    6


### Задание 4.4 — Текстовый отчёт `summary_report.txt`

Сформируйте текстовый аналитический отчёт для руководства с помощью **f-строк**.

Отчёт должен содержать:
- Заголовок и дату
- Общую статистику (всего сообщений, валидных, ошибочных)
- Статистику по типам сообщений
- Статистику по регионам
- Топ-5 публикаторов
- Список приоритетных дел
- Список найденных арбитражных управляющих

In [16]:
# Ваш код
# записываем дату формирования отчёта
reportDate = datetime.now().strftime("%d.%m.%Y %H:%M")

# собираем уникальные ФИО арбитражных управляющих
managers = set()
for m in valid_messages:
    if m.get("manager_name"):
        managers.add(m["manager_name"])

# составляем отчёт построчно через список
lines = []
lines.append("=" * 60)
lines.append("АНАЛИТИЧЕСКИЙ ОТЧЁТ ПО ДАННЫМ ЕФРСБ")
lines.append(f"Дата формирования: {reportDate}")
lines.append("=" * 60)
lines.append("")

lines.append("1. ОБЩАЯ СТАТИСТИКА")
lines.append("-" * 60)
lines.append(f"Всего сообщений на входе:   {len(messages)}")
lines.append(f"Сообщений без ошибками:     {len(valid_messages)}")
lines.append(f"Сообщений с ошибками:       {len(error_records)}")
lines.append(f"Связано с организацией:     {linkedCount}")
lines.append(f"Не связано с организацией:  {notLinkedCount}")
lines.append("")

lines.append("2. СТАТИСТИКА ПО ТИПАМ СООБЩЕНИЙ")
lines.append("-" * 60)
for t, c in sorted(type_stats.items(), key=lambda x: x[1], reverse=True):
    lines.append(f"  {t}: {c}")
lines.append("")

lines.append("3. СТАТИСТИКА ПО РЕГИОНАМ")
lines.append("-" * 60)
for r, c in sorted(region_stats.items(), key=lambda x: x[1], reverse=True):
    lines.append(f"-  {r}: {c}")
lines.append("")

lines.append("4. ТОП-5 ПУБЛИКАТОРОВ")
lines.append("-" * 60)
for i, (inn, count) in enumerate(top_publishers, start=1):
    orgName = org_by_inn.get(inn, {}).get("name", "—")
    lines.append(f"-  {i}. ИНН {inn} | {orgName} | сообщений: {count}")
lines.append("")

lines.append("5. ПРИОРИТЕТНЫЕ ДЕЛА (есть в реестре приоритета и в сообщениях)")
lines.append("-" * 60)
if intersection:
    for caseNumber in sorted(intersection):
        # найдём первое сообщение с этим делом для красивого вывода
        firstMsg = next((m for m in valid_messages if m["case_number"] == caseNumber), None)
        if firstMsg:
            lines.append(f"-  {caseNumber} | {firstMsg['type']} | {firstMsg.get('org_name') or '—'}")
        else:
            lines.append(f"-  {caseNumber}")
else:
    lines.append("  (нет совпадений)")
lines.append("")

lines.append("6. АРБИТРАЖНЫЕ УПРАВЛЯЮЩИЕ (упомянуты в сообщениях)")
lines.append("-" * 60)
if managers:
    for fio in sorted(managers):
        lines.append(f"  - {fio}")
else:
    lines.append("  (не найдено)")
lines.append("")
lines.append("=" * 60)
lines.append("Конец отчёта")
lines.append("=" * 60)
lines.append("")

reportText = "\n".join(lines)

with open(f"{DATA_DIR}/summary_report.txt", "w", encoding="utf-8") as f:
    f.write(reportText)

print()
print(f"Файл summary_report.txt сохранён локально /{DATA_DIR}")
print("   Количество строк записей:   ", len(lines))
print()
for line in lines:
    print(line)


Файл summary_report.txt сохранён локально /final_hw_data
   Количество строк записей:    79

АНАЛИТИЧЕСКИЙ ОТЧЁТ ПО ДАННЫМ ЕФРСБ
Дата формирования: 26.04.2026 08:59

1. ОБЩАЯ СТАТИСТИКА
------------------------------------------------------------
Всего сообщений на входе:   54
Сообщений без ошибками:     48
Сообщений с ошибками:       6
Связано с организацией:     51
Не связано с организацией:  3

2. СТАТИСТИКА ПО ТИПАМ СООБЩЕНИЙ
------------------------------------------------------------
  Введение процедуры: 8
  Завершение процедуры: 6
  Продажа имущества: 6
  Требование кредитора: 4
  Собрание кредиторов: 3
  Признание банкротом: 3
  Оспаривание сделки: 3
  Подача заявления: 3
  Мировое соглашение: 3
  Передача дела: 3
  Субсидиарная ответственность: 2
  Назначение управляющего: 2
  Отстранение управляющего: 1
  Жалоба на управляющего: 1

3. СТАТИСТИКА ПО РЕГИОНАМ
------------------------------------------------------------
-  Москва: 25
-  Свердловская область: 6
-  Санкт-Петербу

In [17]:
# Коммит файлов-результатов в репозиторий git

import os
import json
import base64
import urllib.request
import urllib.error
import ipywidgets as widgets
from IPython.display import display, clear_output

# настройки репозитория
OWNER = "vladislavpykhteev"
REPO = "HSE_Vladislav_Pykhteev"
BRANCH = "main"

# путь_локально -> путь_в_репо
FILES_TO_COMMIT = {
    "final_hw_data/analysis_results.json":  "final/final_hw_data/analysis_results.json",
    "final_hw_data/validation_errors.json": "final/final_hw_data/validation_errors.json",
    "final_hw_data/summary_report.txt":     "final/final_hw_data/summary_report.txt",
}

COMMIT_MESSAGE = "Итоговое ДЗ - результаты"

# работаем с git через API
def github_api(token, method, path, payload=None):
    url = "https://api.github.com" + path
    headers = {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github+json",
        "User-Agent": "hse-final-hw-script",
    }

    data = None
    if payload is not None:
        data = json.dumps(payload).encode("utf-8")
        headers["Content-Type"] = "application/json"

    request = urllib.request.Request(url, data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(request) as response:
            body = response.read().decode("utf-8")
            return response.status, json.loads(body) if body else {}
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        return exc.code, {"Ошибка": body}


def get_existing_sha(token, repo_path):
    status, response = github_api(
        token, "GET", f"/repos/{OWNER}/{REPO}/contents/{repo_path}?ref={BRANCH}"
    )
    if status == 200:
        return response.get("sha")
    return None

# Проверяем есть ли файл, если да - заменяем, если нет - создаем новый
def commit_file(token, local_path, repo_path):
    # работаем с файлом в бинарном виде
    with open(local_path, "rb") as f:
        content_bytes = f.read()
    content_b64 = base64.b64encode(content_bytes).decode("utf-8")

    existing_sha = get_existing_sha(token, repo_path)

    payload = {
        "message": COMMIT_MESSAGE,
        "content": content_b64,
        "branch": BRANCH,
    }
    if existing_sha is not None:
        payload["sha"] = existing_sha

    status, response = github_api(
        token, "PUT", f"/repos/{OWNER}/{REPO}/contents/{repo_path}", payload
    )

    if status in (200, 201):
        action = "обновлён" if existing_sha else "создан"
        commitSha = response.get("commit", {}).get("sha", "")[:7]
        print(f"  OK   {repo_path}  ({action}, commit {commitSha})")
        return True
    else:
        # записываем сообщение об ошибке
        errBody = response.get("Ошибка:", "")
        try:
            errParsed = json.loads(errBody) if isinstance(errBody, str) else errBody
            errMsg = errParsed.get("Ошибка:", str(errBody))
        except Exception:
            errMsg = str(errBody)[:200]
        print(f"  FAIL {repo_path}  status={status}  {errMsg}")
        return False

# Получаем токен для git
def get_token(secret_name):
    # 1) пробуем секреты Colab
    try:
        from google.colab import userdata
        try:
            value = userdata.get(secret_name)
            if value:
                return value, f"секрет Colab '{secret_name}'"
        except Exception:
            pass
    except ImportError:
        pass

    # 2) переменная окружения локального запуска
    envValue = os.environ.get(secret_name, "").strip()
    if envValue:
        return envValue, f"переменная окружения {secret_name}"

    # 3) ручной ввод
    manual = manual_token_input.value.strip()
    if manual:
        return manual, "поле ручного ввода"

    return "", ""

# обрабаытваем нажатие кнопки
def on_commit_click(_):
    with output_area:
        clear_output()
        secretName = secret_name_input.value.strip() or "GH_TOKEN"
        token, source = get_token(secretName)

        if not token:
            print(f"Не нашёл токен в секрете '{secretName}'.")
            print("Проверь:")
            print(f"  - в панели 'Секреты' Colab есть секрет с именем {secretName}")
            print("  - переключатель 'Доступ из блокнотов' включён")
            print("  - либо введи токен в поле ручного ввода ниже и нажми кнопку")
            return

        print(f"Токен взят из: {source}")
        print(f"Коммит файлов в {OWNER}/{REPO}, ветка {BRANCH}")
        print()

        successCount = 0
        failCount = 0
        for localPath, repoPath in FILES_TO_COMMIT.items():
            if not os.path.exists(localPath):
                print(f"  SKIP {repoPath}  (файл {localPath} не найден локально)")
                failCount = failCount + 1
                continue
            ok = commit_file(token, localPath, repoPath)
            if ok:
                successCount = successCount + 1
            else:
                failCount = failCount + 1

        print()
        print(f"Готово. Успешно: {successCount}, ошибок: {failCount}")
        if successCount > 0:
            print(f"Проверьте репозиторий: https://github.com/{OWNER}/{REPO}/tree/{BRANCH}/final")


# создаём виджеты:
# - поле для ввода токена (тип Password - скрывает символы)
# - кнопка запуска
# - область вывода
secret_name_input = widgets.Text(
    value="GH_TOKEN",
    description="Имя секрета:",
    layout=widgets.Layout(width="400px"),
)
btn_commit = widgets.Button(
    description="Закоммитить в GitHub",
    button_style="Primary",
    layout=widgets.Layout(width="400px", height="30px"),
)
output_area = widgets.Output()

btn_commit.on_click(on_commit_click)

# показываем виджеты по порядку
display(secret_name_input)
display(btn_commit)
display(output_area)



Text(value='GH_TOKEN', description='Имя секрета:', layout=Layout(width='400px'))

Button(button_style='primary', description='Закоммитить в GitHub', layout=Layout(height='30px', width='400px')…

Output()

---
## ✅ Итоги

Если вы корректно выполнили все 4 уровня, у вас должны получиться:

| Файл | Описание |
|------|----------|
| `analysis_results.json` | Валидные сообщения + статистика + приоритетные дела |
| `validation_errors.json` | Проблемные записи с описанием ошибок |
| `summary_report.txt` | Текстовый аналитический отчёт для руководства |

